# 🚀 Baidu Unlimited-OCR: Open-Source Scientific PDF-to-Markdown Benchmark on Google Colab
### With Persistent Google Drive Model Caching (Never Re-Download Weights!)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ri5hiPrajapati/pdf-to-markdown-verbatim/blob/main/colab/unlimited_ocr_benchmark.ipynb)

This notebook runs **Baidu's Unlimited-OCR** (`baidu/Unlimited-OCR`) on Google Colab's free **Tesla T4 GPU (15.3 GB VRAM)**.

### Key Highlights:
* **Google Drive Persistence**: Model weights are saved directly to your Google Drive cache. Reconnecting to Colab will load the model in seconds without re-downloading.
* **Model Architecture**: Reference Sliding Window Attention (R-SWA) with constant flat KV cache throughout multi-page autoregressive decoding.

In [ ]:
# Step 1: Verify Hardware Accelerator
!nvidia-smi

In [ ]:
# Step 2: Install Dependencies
!pip install -q transformers>=4.45.0 torch torchvision pymupdf pillow accelerate einops addict easydict psutil markdown

In [ ]:
# Step 3: Mount Google Drive & Configure Persistent Hugging Face Cache
# This ensures model weights (~2.5 GB) are stored permanently in your Drive
# and NEVER need to be re-downloaded when your Colab GPU reconnects!
import os
from google.colab import drive, userdata

print("1. Mounting Google Drive...")
drive.mount('/content/drive')

# Configure persistent cache path in Google Drive
hf_cache_dir = '/content/drive/MyDrive/huggingface_cache'
os.makedirs(hf_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = hf_cache_dir
os.environ['TRANSFORMERS_CACHE'] = hf_cache_dir

# Automatically inject HF_TOKEN if saved in Colab Secrets (silences unauthenticated warnings)
try:
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token.strip()
        print("2. ✅ HF_TOKEN authenticated successfully from Colab Secrets!")
    else:
        print("2. ℹ️ HF_TOKEN not found in Secrets. Using standard public download.")
except Exception:
    print("2. ℹ️ Secrets manager skipped.")

print(f"3. ✅ Persistent Model Cache Active: {hf_cache_dir}")
print("   Weights will download directly to your Google Drive and persist indefinitely!")

In [ ]:
# Step 4: Clone Benchmark Toolkit & Verification Harness
!git clone https://github.com/Ri5hiPrajapati/pdf-to-markdown-verbatim.git
%cd pdf-to-markdown-verbatim

In [ ]:
# Step 5: Load baidu/Unlimited-OCR into GPU Memory
import torch
from transformers import pipeline

model_id = "baidu/Unlimited-OCR"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {model_id} on {device} via Google Drive cache...")

pipe = pipeline("image-text-to-text", model=model_id, trust_remote_code=True, device=0 if torch.cuda.is_available() else -1)
print("✅ Baidu Unlimited-OCR loaded successfully!")

In [ ]:
# Step 6: Define Document Parsing Routine
import io
import time
import pymupdf as fitz
from PIL import Image

def parse_pdf_with_unlimited_ocr(pdf_path, dpi=150):
    doc = fitz.open(pdf_path)
    print(f"Processing: {pdf_path} ({len(doc)} pages)")
    
    page_mds = []
    t0 = time.time()
    
    for idx, page in enumerate(doc):
        pt0 = time.time()
        pix = page.get_pixmap(dpi=dpi)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Extract all text, sections, formulas, and tables from this page verbatim into Markdown."}
                ]
            }
        ]
        
        result = pipe(text=messages)
        out_text = result[0]["generated_text"] if isinstance(result, list) else str(result)
        page_mds.append(out_text)
        print(f"  • Page {idx+1}/{len(doc)}: done in {time.time()-pt0:.2f}s ({len(out_text)} chars)")
        
    total_time = time.time() - t0
    full_markdown = "\n\n---\n\n".join(page_mds)
    out_path = pdf_path.replace(".pdf", "_unlimited_ocr.md")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(full_markdown)
        
    print(f"\n🎉 Conversion complete in {total_time:.2f}s ({total_time/len(doc):.2f}s/page)!")
    print(f"Saved to: {out_path}")
    return out_path

In [ ]:
# Step 7: Upload a Benchmark PDF
from google.colab import files
import shutil

print("Upload your benchmark PDF file:")
uploaded = files.upload()

if uploaded:
    target_pdf = list(uploaded.keys())[0]
else:
    target_pdf = "sample_paper.pdf"
    !wget -q -O sample_paper.pdf "https://arxiv.org/pdf/2401.00001.pdf"

converted_md = parse_pdf_with_unlimited_ocr(target_pdf)

# Preserve a permanent copy in Google Drive
drive_out_dir = '/content/drive/MyDrive/pdf_markdown_outputs'
os.makedirs(drive_out_dir, exist_ok=True)
shutil.copy(converted_md, os.path.join(drive_out_dir, os.path.basename(converted_md)))
print(f"✅ Output safely preserved in Google Drive: {drive_out_dir}/{os.path.basename(converted_md)}")

In [ ]:
# Step 8: Run Quantitative Audit via verify_md.py
!python verify_md.py "{target_pdf}" "{converted_md}"

In [ ]:
# Step 9: Download Converted Markdown
files.download(converted_md)